# Star Graph Pipeline on Kaggle

Runs enrichment + deep research using a local llama.cpp model (no NVIDIA NIM).
Saves results to /kaggle/working/star-graph/data/. GitHub Actions downloads and pushes back.

No secrets needed — the push happens in the GitHub workflow, not here.

In [ ]:
# --- clone both repos ---------------------------------------------------
import subprocess, sys, os, time

for d in ['/kaggle/working/kms', '/kaggle/working/star-graph']:
    subprocess.run(['rm', '-rf', d], capture_output=True)

# Public clones work for public repos
subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/Meru143/kaggle-model-server.git', '/kaggle/working/kms'],
    check=True, timeout=120)
subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/Meru143/star-graph.git', '/kaggle/working/star-graph'],
    check=True, timeout=120)

sys.path.insert(0, '/kaggle/working/kms')
sys.path.insert(0, '/kaggle/working/star-graph/kaggle')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub', 'requests', 'networkx', 'numpy', 'sentence-transformers'],
    check=True, timeout=300)
print('Setup complete')

In [ ]:
# --- imports ------------------------------------------------------------
import importlib
for _m in ('model_registry', 'harness'):
    if _m in sys.modules:
        if _m == 'harness':
            try: sys.modules['harness'].stop()
            except Exception: pass
        importlib.reload(sys.modules[_m])

from model_registry import MODELS
from harness import run, stop, harvest_cache
from star_graph_kaggle import run_pipeline

print(f'Models: {list(MODELS.keys())}')

In [ ]:
# --- boot the local LLM -------------------------------------------------
# First run: downloads GGUF + builds llama.cpp (~15 min)
# Cached run: ~2 min (after harvest_cache done once)

url = run('owao/Nanbeige4.2-3B-GGUF', MODELS, quant='Q4_K_M', ctx=4096)
print(f'Model ready at {url}')

In [ ]:
# --- run the pipeline (no NVIDIA calls, all local) ----------------------
run_pipeline(limit=None)
print('Pipeline complete. Data saved to /kaggle/working/star-graph/data/')

In [ ]:
# --- cache binaries for faster next boot --------------------------------
try:
    harvest_cache()
    print('Cache harvested')
except Exception as e:
    print(f'Cache harvest skipped: {e}')

In [ ]:
# --- clean shutdown -----------------------------------------------------
stop()
print('Model stopped. Session ending.')